# FLAG 2027 - Notebook 15: Stack the cheap wins on top of B1x (dev 29.45)

| Version | Idea | Source |
|---|---|---|
| B0 | B1x exactly (reference, rerun for a paired comparison) | Notebook 14 |
| P1 | **fixed maximum-separation prototypes** as classifier + OPL | Moscati et al. ICASSP 2026 (organizers' method): 23.8 -> 22.9 |
| P2 | **gender-balanced batches** (8 men + 8 women) so same-gender negatives are always present | literature + our same-gender weakness |
| P1+P2 | both | |
| +G | **gender-consistency term** on no_gender trials: score - lambda x \|P(female\|face) - P(female\|voice)\|, lambda chosen on TRAINING validation only (explicitly allowed in the organizers' FAQ) | FAQ 21 Oct; FAME24 #3 |

Gender for MAV-Celeb people (no labels) is predicted per person from the face with a classifier trained on labelled TRAINING data (challenge + VoxCeleb2).

About 1.5-2 h. Resumable. Paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive + GPU
from google.colab import drive
drive.mount('/content/drive')
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
# Cell 2 - load challenge + MAV v1/v3 + MAV extra (v2 + v1 held-out) + VoxCeleb2 (as Notebook 14)
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve
from sklearn.linear_model import LogisticRegression

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
RESDIR = os.path.join(PROJECT, 'results')
SCOREDIR = os.path.join(RESDIR, 'nb15_scores'); os.makedirs(SCOREDIR, exist_ok=True)
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_small'; os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FEAT = {}
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
FEAT['ecapa'] = {'train': Xv}
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1)
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang_ + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv', 'r293']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); MAV['r293'] = FEAT['r293']['mav']
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
VOX = {k: [] for k in ['spk', 'gender', 'clip', 'ecapa', 'wavlm_sv', 'r293']}
for tag in ['vox2_test_mp4', 'vox2_mp4_1']:
    d = dict(np.load(os.path.join(FEATDIR, 'vox_' + tag + '.npz'))); d['r293'] = np.load(os.path.join(FEATDIR, 'vox_' + tag + '_r293.npy'))
    for k in VOX:
        VOX[k].append(d[k])
VOX = {k: np.concatenate(v) for k, v in VOX.items()}

def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
VOICE = ['ecapa', 'wavlm_sv', 'r293']
def face_of(clip):
    return l2(clip).astype(np.float32)
def voice_of(src):
    return np.concatenate([l2(src[n]) for n in VOICE], 1).astype(np.float32)
CH = {n: FEAT[n]['train'] for n in ['clip'] + VOICE}
N_CH, N_MAV, N_EX, N_VOX = len(y), len(MAV['spk']), len(EXTRA['spk']), len(VOX['spk'])
ext_spk = list(MAV['spk']) + list(EXTRA['spk']) + list(VOX['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in ext_spk])])
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV + N_EX, int), 2 * np.ones(N_VOX, int)]
F_RAW = np.concatenate([face_of(CH['clip']), face_of(MAV['clip']), face_of(EXTRA['clip']), face_of(VOX['clip'])])
V_RAW = np.concatenate([voice_of(CH), voice_of(MAV), voice_of(EXTRA), voice_of(VOX)])
EXT = N_CH + np.arange(N_MAV + N_EX + N_VOX)

# gender per row: challenge + VoxCeleb2 have labels; MAV people -> predicted per person from the face (TRAINING data only)
def gcode(g):
    g = str(g).strip().lower(); return 1 if g.startswith('f') else (0 if g.startswith('m') else -1)
G_CH = np.array([gcode(id_gender[c]) for c in y]); G_VOX = np.array([gcode(g) for g in VOX['gender']])
lab = G_VOX >= 0
gclf_face = LogisticRegression(max_iter=3000, C=0.1).fit(np.r_[F_RAW[:N_CH], face_of(VOX['clip'])[lab]], np.r_[G_CH, G_VOX[lab]])
mav_clip = np.concatenate([face_of(MAV['clip']), face_of(EXTRA['clip'])]); mav_spk = np.r_[MAV['spk'], EXTRA['spk']]
pp = pd.Series(gclf_face.predict(mav_clip)).groupby(mav_spk).mean()
G_MAV = np.array([int(pp[s] >= 0.5) for s in mav_spk])
G_ALL = np.r_[G_CH, G_MAV, G_VOX]
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
note('Table: %d rows, %d people | MAV people predicted female: %d of %d' % (len(Y_ALL), len(set(Y_ALL)), int((pp >= 0.5).sum()), len(pp)))

In [ ]:
# Cell 3 - model (+ optional fixed max-separation prototypes), losses, validation
def domain_normalize(tr_idx):
    F, V = F_RAW.copy(), V_RAW.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_RAW[trd].mean(0), F_RAW[trd].std(0) + 1e-6, V_RAW[trd].mean(0), V_RAW[trd].std(0) + 1e-6)
        F[alld] = (F_RAW[alld] - st[0]) / st[1]; V[alld] = (V_RAW[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch

_PROTO = {}
def max_sep_prototypes(n):
    # Kasarla et al. 2022: n unit vectors in R^(n-1), all pairwise cosines = -1/(n-1) (maximally separated). Returns (n, n-1).
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P
        P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32)
    return _PROTO[n]

class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128, proto=False):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.proto = proto
        if proto:   # learnable map to R^(n-1), then FIXED prototypes give the logits
            self.cls = nn.Linear(d, n_class - 1)
            self.register_buffer('P', max_sep_prototypes(n_class))
        else:
            self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        logits = self.cls(fused) @ self.P.t() if self.proto else self.cls(fused)
        return fused, logits
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
print('ready')

In [ ]:
# Cell 4 - training (B1x recipe) with options: proto (P1), gender-balanced batches (P2); RESUMABLE; scores saved
SEEDS = [1, 2, 3]
P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS, IN_FRAC = 16, 8, 2.0, 1e-4, 100, 15, 60, 0.75

def pick(rng, pool, n, cls_g, gbal):
    if not gbal:
        return rng.choice(pool, n, replace=False)
    m, f = pool[cls_g[pool] == 0], pool[cls_g[pool] == 1]
    nf = n // 2
    if len(f) < nf or len(m) < n - nf:
        return rng.choice(pool, n, replace=False)
    return np.r_[rng.choice(m, n - nf, replace=False), rng.choice(f, nf, replace=False)]

def train_variant(tr_idx, proto=False, gbal=False, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch = domain_normalize(all_tr)
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev)
    cls_in = np.zeros(len(cl), bool); cls_in[np.unique(ytr_np[D_ALL[all_tr] < 2])] = True
    cls_g = np.zeros(len(cl), int)
    for c_, g_ in zip(ytr_np, G_ALL[all_tr]):
        cls_g[c_] = g_
    in_cls, out_cls = np.where(cls_in)[0], np.where(~cls_in)[0]; n_in = int(round(P_ID * IN_FRAC))
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev)
    model = FOP(len(cl), XF.shape[1], XV.shape[1], proto=proto).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)
    mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = np.r_[pick(rng, in_cls, n_in, cls_g, gbal), pick(rng, out_cls, P_ID - n_in, cls_g, gbal)]
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch

VARIANTS = {'B0_b1x': (False, False), 'P1_proto': (True, False), 'P2_genderbal': (False, True), 'P12_both': (True, True)}
CSV = os.path.join(RESDIR, 'nb15_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['variant', 'seed', 'fold'])
t0 = time.time()
for name, (pr, gb) in VARIANTS.items():
    for seed in SEEDS:
        for k in range(K):
            if ((done.variant == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep, XF, XV, _ = train_variant(tr_idx, pr, gb, vsets, seed=seed)
            sc = [same_score(model, XF[Fi], XV[Vi]) for Vi, Fi, L in vsets]
            np.savez(os.path.join(SCOREDIR, '%s_s%d_f%d.npz' % (name, seed, k)), std=sc[0], gen=sc[1])
            e_std, e_gen = [eer(L, s) for (Vi, Fi, L), s in zip(vsets, sc)]
            row = pd.DataFrame([dict(variant=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-13s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min' % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))
R = pd.read_csv(CSV)
R['overall'] = (R['std'] + R['gender']) / 2
tab = R.groupby(['variant', 'seed'])[['std', 'gender', 'overall']].mean().groupby('variant').mean()
tab = tab.loc[[v for v in VARIANTS if v in tab.index]].round(2)
note('STACKED WINS (CV 5-fold x 3 seeds; B1x in Notebook 14 = 25.92)')
note(tab.to_string())

In [ ]:
# Cell 5 - gender-consistency term (FAQ-allowed): s' = s - lam * |P(f|face) - P(f|voice)| on no_gender trials only.
# Gender classifiers are trained on each fold's TRAINING rows; lam is chosen on the validation folds (training people).
LAMS = [0.0, 0.1, 0.2, 0.3, 0.5, 0.8]
def gender_probs(tr_rows, Xf_rows, Xv_rows):
    labd = tr_rows[G_ALL[tr_rows] >= 0]
    cf = LogisticRegression(max_iter=3000, C=0.1).fit(F_RAW[labd], G_ALL[labd])
    cv_ = LogisticRegression(max_iter=3000, C=0.1).fit(V_RAW[labd], G_ALL[labd])
    return cf, cv_
best_var = tab['overall'].idxmin()
res = {l: [] for l in LAMS}
for k in range(K):
    tr_idx, vsets = FOLD_DATA[k]
    cf, cv_ = gender_probs(np.concatenate([tr_idx, EXT]), None, None)
    Vi, Fi, L = vsets[0]                                    # standard (no_gender-like) validation pairs
    gap = np.abs(cf.predict_proba(F_RAW[Fi])[:, 1] - cv_.predict_proba(V_RAW[Vi])[:, 1])
    for s in SEEDS:
        base = np.load(os.path.join(SCOREDIR, '%s_s%d_f%d.npz' % (best_var, s, k)))['std']
        for l in LAMS:
            res[l].append(eer(L, base - l * gap))
G_TAB = pd.Series({l: np.mean(v) for l, v in res.items()}).round(2)
LAM = float(G_TAB.idxmin())
note('Gender-consistency term on %s, standard-pair EER by lambda: %s -> chosen lambda %.1f' % (best_var, G_TAB.to_dict(), LAM))

In [ ]:
# Cell 6 - final model of the best variant (+ gender-consistency term if it helped) -> submission ZIP + dev scores
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
pr, gb = VARIANTS[best_var]
ep = int(np.median(R[R.variant == best_var]['best_ep']))
finals = []
for s in SEEDS:
    m, _, _, _, ch = train_variant(np.arange(N_CH), pr, gb, None, seed=s, max_epochs=ep)
    finals.append(m)
cf, cv_ = gender_probs(np.concatenate([np.arange(N_CH), EXT]), None, None)   # TRAINING rows only
sc = {}
for track in ['no_gender', 'gender']:
    for lang_ in LANG_FILE:
        split = track + '_' + lang_
        fx = ((face_of(FEAT['clip'][split]) - ch[0]) / ch[1]).astype(np.float32)
        vx = ((voice_of({n: FEAT[n][split] for n in VOICE}) - ch[2]) / ch[3]).astype(np.float32)
        xf = torch.tensor(fx, device=dev); xv = torch.tensor(vx, device=dev)
        s_ = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
        if track == 'no_gender' and LAM > 0:     # fixed term, lambda chosen on TRAINING validation; per-trial only
            # raw (un-normalised) features, exactly like the classifier's training input
            fr = face_of(FEAT['clip'][split]); vr = voice_of({n: FEAT[n][split] for n in VOICE})
            # the classifiers were fitted on F_RAW/V_RAW rows = raw features of the CHALLENGE domain -> same space
            s_ = s_ - LAM * np.abs(cf.predict_proba(fr)[:, 1] - cv_.predict_proba(vr)[:, 1])
        sc[split] = s_
name = '15_%s%s' % (best_var, '_gcons%.1f' % LAM if LAM > 0 else '')
OUT = '/content/sub_' + name
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang_, fname in LANG_FILE.items():
        split = track + '_' + lang_
        pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang_ + '_test.txt')) if l.strip()]
        assert len(pair_ids) == len(sc[split]) and np.isfinite(sc[split]).all()
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, -sc[split]):
                f.write('%s %.6f\n' % (pid, v))
zp = os.path.join(SUBDIR, name.replace('.', 'p') + '.zip')
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
np.savez(os.path.join(RESDIR, 'devscores_%s.npz' % name.replace('.', 'p')), **sc)
note('Submission ZIP saved: ' + zp[len(BASE):])
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)